# ResNet18 EMG Gesture Classification
Compares **colour** vs **black-and-white** image representations.

Two evaluation protocols:
- **Intra-subject**: model trained on some trials per participant, tested on held-out trials from the *same* participants
- **Inter-subject**: model trained on some participants entirely, tested on *unseen* participants

**Before running**: upload both dataset folders to Google Drive and set the paths in Cell 2.

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1b: Unzip datasets (run once) ────────────────────────────────────────
# Upload new_dataset_resnet_local.zip and new_dataset_resnet_bw.zip to the root
# of your Google Drive, then run this cell once to extract them.
import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive'

for zip_name in ['new_dataset_resnet_local.zip', 'new_dataset_resnet_bw.zip']:
    zip_path   = f'{DRIVE_ROOT}/{zip_name}'
    folder_name = zip_name.replace('.zip', '')
    out_path   = f'{DRIVE_ROOT}/{folder_name}'

    if os.path.exists(out_path):
        print(f'Already extracted: {folder_name}')
        continue
    print(f'Extracting {zip_name} …')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DRIVE_ROOT)
    print(f'  → {out_path}')

In [ ]:
# ── Cell 2: Configuration — edit these paths ───────────────────────────────────

# Paths to the two dataset folders in your Drive
# Each folder should contain subfolders: 3-finger_extension, 3-finger_flexion, etc.
COLOUR_DIR = '/content/drive/MyDrive/new_dataset_resnet_local'
BW_DIR     = '/content/drive/MyDrive/new_dataset_resnet_bw'

# Where to save results
RESULTS_DIR = '/content/drive/MyDrive/resnet18_results'

BATCH_SIZE  = 64
NUM_EPOCHS  = 25
LR          = 1e-4
PATIENCE    = 5      # early stopping patience (epochs)
SEED        = 42

# Inter-subject split: these subject numbers are held out entirely for test
# (roughly the last ~20% of participants)
INTER_TEST_SUBJECTS  = {30, 31, 32, 33, 34, 35, 36}   # 7 subjects
INTER_VAL_SUBJECTS   = {25, 26, 27, 28, 29}            # 5 subjects
# train = subjects 1-24

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import os, re, json, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 4: File scanning and metadata extraction ──────────────────────────────

CLASS_NAMES = sorted([
    '3-finger_extension', '3-finger_flexion',
    'index_extension',    'index_flexion',
    'wrist_extension',    'wrist_flexion',
])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def parse_filename(fname):
    """
    From a filename like:
      (02)Bosco_da2-(00)trial 3-(00)wrist_flexion_j0_shift00_rshift20.png
    Returns:
      subject_num (int): 2
      trial_num   (int): 3
      is_unshifted (bool): True if shift00_rshift00
    """
    m_subj  = re.match(r'\((\d+)\)', fname)
    m_trial = re.search(r'trial (\d+)', fname)
    is_unshifted = '_shift00_rshift00' in fname
    subject_num = int(m_subj.group(1))  if m_subj  else -1
    trial_num   = int(m_trial.group(1)) if m_trial else -1
    return subject_num, trial_num, is_unshifted

def scan_dataset(root):
    """
    Returns a list of dicts:
      {path, label, subject_num, trial_num, is_unshifted}
    """
    records = []
    root = Path(root)
    for class_name in CLASS_NAMES:
        class_dir = root / class_name
        if not class_dir.exists():
            print(f'WARNING: missing class folder {class_dir}')
            continue
        for img_path in sorted(class_dir.glob('*.png')):
            subj, trial, is_unshifted = parse_filename(img_path.name)
            records.append({
                'path':         str(img_path),
                'label':        CLASS_TO_IDX[class_name],
                'subject_num':  subj,
                'trial_num':    trial,
                'is_unshifted': is_unshifted,
            })
    print(f'Scanned {root.name}: {len(records):,} images')
    return records

colour_records = scan_dataset(COLOUR_DIR)
bw_records     = scan_dataset(BW_DIR)

In [ ]:
# ── Cell 5: Split strategies ───────────────────────────────────────────────────

def intra_subject_split(records):
    """
    All participants appear in train/val/test.
    For each participant:
      - last trial  → test  (unshifted only)
      - second-last → val   (unshifted only)
      - all others  → train (all 9 shift variants)
    """
    # Find max trial number per subject
    subj_trials = defaultdict(set)
    for r in records:
        subj_trials[r['subject_num']].add(r['trial_num'])

    train, val, test = [], [], []
    for r in records:
        s = r['subject_num']
        t = r['trial_num']
        trials = sorted(subj_trials[s])
        last   = trials[-1]
        second = trials[-2] if len(trials) > 1 else trials[-1]

        if t == last:
            if r['is_unshifted']:  test.append(r)
        elif t == second:
            if r['is_unshifted']:  val.append(r)
        else:
            train.append(r)

    return train, val, test


def inter_subject_split(records):
    """
    Subjects 30-36 → test  (unshifted only)
    Subjects 25-29 → val   (unshifted only)
    Subjects  1-24 → train (all 9 shift variants)
    """
    train, val, test = [], [], []
    for r in records:
        s = r['subject_num']
        if s in INTER_TEST_SUBJECTS:
            if r['is_unshifted']:  test.append(r)
        elif s in INTER_VAL_SUBJECTS:
            if r['is_unshifted']:  val.append(r)
        else:
            train.append(r)

    return train, val, test


def split_summary(train, val, test, name):
    print(f'\n── {name} ──')
    print(f'  Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}')
    for split_name, split in [('Train', train), ('Val', val), ('Test', test)]:
        counts = defaultdict(int)
        for r in split:
            counts[CLASS_NAMES[r['label']]] += 1
        print(f'  {split_name}: ' + ', '.join(f'{k.split("_")[0]}_{k.split("_")[1][:3]}={v}' for k,v in sorted(counts.items())))

# Preview splits on colour dataset
tr, va, te = intra_subject_split(colour_records)
split_summary(tr, va, te, 'Intra-subject')

tr, va, te = inter_subject_split(colour_records)
split_summary(tr, va, te, 'Inter-subject')

In [ ]:
# ── Cell 6: Dataset class ──────────────────────────────────────────────────────

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

class EMGDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r   = self.records[idx]
        img = Image.open(r['path']).convert('RGB')
        return self.transform(img), r['label']

def make_loaders(train_r, val_r, test_r, batch_size=BATCH_SIZE):
    train_ds = EMGDataset(train_r, TRANSFORM_TRAIN)
    val_ds   = EMGDataset(val_r,   TRANSFORM_EVAL)
    test_ds  = EMGDataset(test_r,  TRANSFORM_EVAL)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_dl, val_dl, test_dl

In [ ]:
# ── Cell 7: Model ──────────────────────────────────────────────────────────────

def make_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(512, len(CLASS_NAMES))
    return model.to(device)

In [ ]:
# ── Cell 8: Training loop ──────────────────────────────────────────────────────

def train_model(train_dl, val_dl, run_name):
    model     = make_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    best_val_acc  = 0.0
    best_weights  = None
    no_improve    = 0
    history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        # ── train ──
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * imgs.size(0)
            correct  += (out.argmax(1) == labels).sum().item()
            total    += imgs.size(0)
        train_loss = run_loss / total
        train_acc  = correct  / total

        # ── val ──
        model.eval()
        run_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(device), labels.to(device)
                out  = model(imgs)
                loss = criterion(out, labels)
                run_loss += loss.item() * imgs.size(0)
                correct  += (out.argmax(1) == labels).sum().item()
                total    += imgs.size(0)
        val_loss = run_loss / total
        val_acc  = correct  / total

        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'[{run_name}] Epoch {epoch:02d}/{NUM_EPOCHS}  '
              f'train_loss={train_loss:.4f} train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.4f} val_acc={val_acc:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_weights)
    print(f'  Best val acc: {best_val_acc:.3f}')
    return model, history

In [ ]:
# ── Cell 9: Evaluation ─────────────────────────────────────────────────────────

def evaluate(model, test_dl):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return acc, all_labels, all_preds


def plot_results(history, labels, preds, run_name, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(run_name, fontsize=14)

    # Training curves
    ax = axes[0]
    ax.plot(history['train_acc'], label='Train acc')
    ax.plot(history['val_acc'],   label='Val acc')
    ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.legend()

    ax = axes[1]
    ax.plot(history['train_loss'], label='Train loss')
    ax.plot(history['val_loss'],   label='Val loss')
    ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.legend()

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    short = [c.replace('3-finger', '3f').replace('index', 'idx').replace('wrist', 'wr')
               .replace('_flexion', '_fl').replace('_extension', '_ex') for c in CLASS_NAMES]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=short, yticklabels=short, ax=axes[2])
    axes[2].set_title('Confusion matrix')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')

    plt.tight_layout()
    safe_name = run_name.replace(' ', '_').replace('/', '-')
    plt.savefig(f'{save_dir}/{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 10: Run all 4 experiments ────────────────────────────────────────────
#
# Experiment matrix:
#   colour  × intra-subject
#   colour  × inter-subject
#   B&W     × intra-subject
#   B&W     × inter-subject

experiments = [
    ('Colour – Intra-subject', colour_records, intra_subject_split),
    ('Colour – Inter-subject', colour_records, inter_subject_split),
    ('B&W   – Intra-subject', bw_records,     intra_subject_split),
    ('B&W   – Inter-subject', bw_records,     inter_subject_split),
]

results = []

for run_name, records, split_fn in experiments:
    print(f'\n{'='*60}')
    print(f'  {run_name}')
    print(f'{'='*60}')

    train_r, val_r, test_r = split_fn(records)
    split_summary(train_r, val_r, test_r, run_name)

    train_dl, val_dl, test_dl = make_loaders(train_r, val_r, test_r)

    model, history = train_model(train_dl, val_dl, run_name)

    test_acc, labels, preds = evaluate(model, test_dl)
    print(f'\nTest accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

    plot_results(history, labels, preds, run_name, RESULTS_DIR)

    # Per-class accuracy
    cm = confusion_matrix(labels, preds)
    per_class_acc = cm.diagonal() / cm.sum(axis=1)

    results.append({
        'Experiment':  run_name,
        'Test Acc':    round(test_acc, 4),
        **{CLASS_NAMES[i]: round(per_class_acc[i], 4) for i in range(len(CLASS_NAMES))},
    })

print('\nAll experiments complete.')

In [ ]:
# ── Cell 11: Summary comparison table ─────────────────────────────────────────

df = pd.DataFrame(results).set_index('Experiment')
print('\n=== RESULTS SUMMARY ===')
print(df.to_string())

# Save to CSV
df.to_csv(f'{RESULTS_DIR}/summary.csv')
print(f'\nSaved to {RESULTS_DIR}/summary.csv')

# Plot overall accuracy comparison
fig, ax = plt.subplots(figsize=(10, 5))
df['Test Acc'].plot(kind='bar', ax=ax, color=['#e74c3c','#e74c3c','#2c3e50','#2c3e50'],
                    alpha=0.8, edgecolor='black')
ax.set_ylabel('Test Accuracy')
ax.set_title('ResNet18 — Colour vs B&W, Intra vs Inter-subject')
ax.set_ylim(0, 1)
ax.axhline(1/6, color='gray', linestyle='--', label='Chance (16.7%)')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()